# Acoustic features vs. speaker age — full population



In [ ]:
import csv
import os
import sys
import time
import pickle
import subprocess
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from threadpoolctl import threadpool_limits

# ── paths: adjust to your setup ──────────────────────────────────────────────
PAPERUTILS_DIR    = Path("..")                                     # contains paperutils/sv2.py
ENRICHMENT_DIR    = Path("datasets/voxceleb_enrichment_age_gender")  # age/gender labels for VoxCeleb2
VOXCELEB2_M4A_DIR = "/path/to/VoxCeleb2/"                          # original .m4a recordings
VOXCELEB2_WAV_DIR = "/path/to/VoxCeleb2_wav/"                      # same tree, converted to 16 kHz .wav
OUTPUT_DIR        = Path("outputs")

# final_dataframe_extended: https://github.com/hechmik/voxceleb_enrichment_age_gender/tree/main/dataset
ENRICHMENT_CSV   = ENRICHMENT_DIR / "dataset" / "final_dataframe_extended.csv"
RECORDINGS_CSV   = OUTPUT_DIR / "data_cp.csv"
FEATURES_PARQUET = OUTPUT_DIR / "features_pool_5s.parquet"
CORRELATIONS_CSV = OUTPUT_DIR / "corr_abs_pearson_full_population.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Build the recording list


In [ ]:
sys.path.insert(0, str(PAPERUTILS_DIR))
from paperutils.sv2 import get_all_paths

if RECORDINGS_CSV.exists():
    print(f"exists, not recomputed: {RECORDINGS_CSV}")
else:
    enrichment = pd.read_csv(ENRICHMENT_CSV, low_memory=False)
    recordings = get_all_paths(enrichment, VOXCELEB2_M4A_DIR)
    recordings = recordings[["VoxCeleb_ID", "video_id", "gender", "speaker_age", "recording_path"]].dropna()
    recordings["dest"] = recordings["recording_path"].str.replace(VOXCELEB2_M4A_DIR, VOXCELEB2_WAV_DIR, regex=False)
    recordings.to_csv(RECORDINGS_CSV)
    print(f"saved {len(recordings):,} rows -> {RECORDINGS_CSV}")

## Convert `.m4a` → `.wav`

Every `recording_path` of the recording list is converted with ffmpeg to 16 kHz mono 16-bit PCM at `dest` (with `.m4a` → `.wav`)

In [ ]:
CONVERT_N_PROC = 16
CONVERT_PROGRESS_EVERY = 200
CONVERT_FAILURE_LOG = OUTPUT_DIR / "convert_failures.csv"


def convert_one(task):
    m4a_path, dest = task
    wav_path = Path(dest).with_suffix(".wav")
    if wav_path.exists() and wav_path.stat().st_size > 0:
        return m4a_path, None
    wav_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-nostats",
           "-i", m4a_path,
           "-ar", "16000", "-ac", "1", "-sample_fmt", "s16",
           str(wav_path)]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    except subprocess.TimeoutExpired:
        return m4a_path, "timeout (>120s)"
    except Exception as e:
        return m4a_path, f"exception: {e}"
    if result.returncode != 0:
        return m4a_path, f"ffmpeg rc={result.returncode}: {result.stderr.strip()[:200]}"
    if not wav_path.exists() or wav_path.stat().st_size == 0:
        return m4a_path, "wav missing or empty after ffmpeg"
    return m4a_path, None


def convert_all():
    recordings = pd.read_csv(RECORDINGS_CSV)
    tasks = list(zip(recordings["recording_path"], recordings["dest"]))
    n_total = len(tasks)
    print(f"[load] {RECORDINGS_CSV}  rows: {n_total}", flush=True)

    in_source_dir = [d for _, d in tasks if d.startswith(VOXCELEB2_M4A_DIR)]
    if in_source_dir:
        raise RuntimeError(f"{len(in_source_dir)} dest paths under VOXCELEB2_M4A_DIR")

    t_start = time.time()
    n_done, n_failed = 0, 0
    with mp.Pool(CONVERT_N_PROC) as pool, open(CONVERT_FAILURE_LOG, "w", newline="") as fail_file:
        fail_log = csv.writer(fail_file)
        fail_log.writerow(["m4a_path", "error"])
        for m4a_path, error in pool.imap_unordered(convert_one, tasks, chunksize=8):
            n_done += 1
            if error:
                n_failed += 1
                fail_log.writerow([m4a_path, error])
                fail_file.flush()
            if n_done % CONVERT_PROGRESS_EVERY == 0 or n_done == n_total:
                elapsed = time.time() - t_start
                print(f"[{n_done:6d}/{n_total}] ok={n_done - n_failed} fail={n_failed} "
                      f"elapsed={elapsed/60:5.1f}min", flush=True)
    print(f"[done] total={n_total} ok={n_total - n_failed} failed={n_failed}", flush=True)


convert_all()

## Feature extractor — librosa, parselmouth, scipy, numpy


In [ ]:
import numpy as np
import librosa
import parselmouth
from parselmouth.praat import call
import scipy.signal as ssignal
from scipy.signal import find_peaks, butter, filtfilt, hilbert
from scipy.stats import entropy
from scipy.ndimage import uniform_filter

# ── constants ────────────────────────────────────────────────────────────────────
SR           = 16000
N_FFT        = 1024
HOP          = 256
N_MFCC       = 13
F0_MIN       = 75
F0_MAX       = 600
MAX_FORMANTS = 5
MAX_FREQ     = 5500

# EXT (Tier 1–3)
BAND_EDGES   = [0, 500, 1000, 2000, 4000, 8000]   # Hz, eGeMAPS-style
N_FFT_FINE   = 2048   # finer frequency resolution for SHR / H1-H2 / harmonics

# Praat jitter / shimmer arguments: time range (whole file), shortest period,
# longest period, max period factor (+ max amplitude factor for shimmer)
_JITTER_ARGS  = (0, 0, 0.0001, 0.02, 1.3)
_SHIMMER_ARGS = (0, 0, 0.0001, 0.02, 1.3, 1.6)


# ═══════════════════════════════════════════════════════════════════════════════
# 1. FEATS (100) — jitter / shimmer / HNR / formants / MFCC
# ═══════════════════════════════════════════════════════════════════════════════

def _jitter(snd):
    try:
        pp = call(snd, "To PointProcess (periodic, cc)", F0_MIN, F0_MAX)
        return dict(
            jitter_local     = call(pp, "Get jitter (local)",           *_JITTER_ARGS),
            jitter_local_abs = call(pp, "Get jitter (local, absolute)", *_JITTER_ARGS),
            jitter_rap       = call(pp, "Get jitter (rap)",             *_JITTER_ARGS),
            jitter_ppq5      = call(pp, "Get jitter (ppq5)",            *_JITTER_ARGS),
            jitter_ddp       = call(pp, "Get jitter (ddp)",             *_JITTER_ARGS),
        )
    except Exception:
        return {k: np.nan for k in
                ["jitter_local", "jitter_local_abs", "jitter_rap", "jitter_ppq5", "jitter_ddp"]}


def _shimmer(snd):
    try:
        pp  = call(snd, "To PointProcess (periodic, cc)", F0_MIN, F0_MAX)
        loc = call([snd, pp], "Get shimmer (local)",  *_SHIMMER_ARGS)
        try:
            loc_db = call([snd, pp], "Get shimmer (local, dB)", *_SHIMMER_ARGS)
        except Exception:
            loc_db = float(20 * np.log10(1 + loc)) if np.isfinite(loc) else np.nan
        return dict(
            shimmer_local    = loc,
            shimmer_apq3     = call([snd, pp], "Get shimmer (apq3)",  *_SHIMMER_ARGS),
            shimmer_apq5     = call([snd, pp], "Get shimmer (apq5)",  *_SHIMMER_ARGS),
            shimmer_apq11    = call([snd, pp], "Get shimmer (apq11)", *_SHIMMER_ARGS),
            shimmer_dda      = call([snd, pp], "Get shimmer (dda)",   *_SHIMMER_ARGS),
            shimmer_local_dB = loc_db,
        )
    except Exception:
        return {k: np.nan for k in
                ["shimmer_local", "shimmer_local_dB", "shimmer_apq3",
                 "shimmer_apq5", "shimmer_apq11", "shimmer_dda"]}


def _hnr(snd):
    try:
        h = call(snd, "To Harmonicity (cc)", 0.01, F0_MIN, 0.1, 1.0)
        return dict(
            hnr_mean = call(h, "Get mean",               0, 0),
            hnr_std  = call(h, "Get standard deviation", 0, 0),
        )
    except Exception:
        return dict(hnr_mean=np.nan, hnr_std=np.nan)


def _formants(snd):
    try:
        f = snd.to_formant_burg(max_number_of_formants=MAX_FORMANTS,
                                maximum_formant=MAX_FREQ)
        vals = {n: [] for n in range(1, 4)}
        for t in f.xs():
            for n in range(1, 4):
                v = f.get_value_at_time(n, t)
                if np.isfinite(v) and v > 0:
                    vals[n].append(v)
        out = {}
        for n in range(1, 4):
            a = np.array(vals[n])
            k = f"F{n}"
            out[f"{k}_mean"]   = float(np.mean(a))   if len(a) else np.nan
            out[f"{k}_std"]    = float(np.std(a))    if len(a) else np.nan
            out[f"{k}_median"] = float(np.median(a)) if len(a) else np.nan
        return out
    except Exception:
        return {f"F{n}_{s}": np.nan
                for n in range(1, 4) for s in ("mean", "std", "median")}


def _mfcc(y):
    mfcc  = librosa.feature.mfcc(y=y.astype(np.float32), sr=SR, n_mfcc=N_MFCC)
    d     = librosa.feature.delta(mfcc)
    dd    = librosa.feature.delta(mfcc, order=2)
    out   = {}
    for i in range(N_MFCC):
        p = f"mfcc{i+1:02d}"
        out[f"{p}_mean"]    = float(np.mean(mfcc[i]))
        out[f"{p}_std"]     = float(np.std(mfcc[i]))
        out[f"d_{p}_mean"]  = float(np.mean(d[i]))
        out[f"d_{p}_std"]   = float(np.std(d[i]))
        out[f"dd_{p}_mean"] = float(np.mean(dd[i]))
        out[f"dd_{p}_std"]  = float(np.std(dd[i]))
    return out


# ═══════════════════════════════════════════════════════════════════════════════
# 2. DIRECT (25) — env / mod / frame variability / aci
# ═══════════════════════════════════════════════════════════════════════════════

def _direct(y, S):
    rms       = librosa.feature.rms(y=y, hop_length=HOP)[0]
    centroid  = librosa.feature.spectral_centroid(S=S, sr=SR)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=SR)[0]
    rolloff   = librosa.feature.spectral_rolloff(S=S, sr=SR)[0]
    zcr       = librosa.feature.zero_crossing_rate(y, hop_length=HOP)[0]

    def _frame_stats(v, name):
        dx = np.diff(v)
        cv = float(np.std(v) / (np.abs(np.mean(v)) + 1e-9))
        return {
            f"{name}_mean":      float(np.mean(v)),
            f"{name}_std":       float(np.std(v)),
            f"{name}_cv":        cv,
            f"{name}_delta_std": float(np.std(dx)),
        }

    out = {}
    out.update(_frame_stats(rms,       "rms"))
    out.update(_frame_stats(centroid,  "centroid"))
    out.update(_frame_stats(bandwidth, "bandwidth"))
    out.update(_frame_stats(rolloff,   "rolloff"))
    out.update(_frame_stats(zcr,       "zcr"))

    env    = np.abs(ssignal.hilbert(y))
    env_ds = ssignal.resample(env, max(1, len(env) // HOP))
    out["env_mean"] = float(np.mean(env_ds))
    out["env_std"]  = float(np.std(env_ds))

    mod_spec        = np.abs(np.fft.rfft(rms))
    out["mod_std"]  = float(np.std(mod_spec))
    out["mod_entropy"] = float(entropy(mod_spec / (np.sum(mod_spec) + 1e-9)))

    diff = np.abs(np.diff(S, axis=1))
    out["aci"] = float(np.sum(diff) / (np.sum(S) + 1e-9))

    return out, rms


# ═══════════════════════════════════════════════════════════════════════════════
# 3. EXTRA (17) — F0 / formant bandwidths / CPP / spectral tilt
# ═══════════════════════════════════════════════════════════════════════════════

def _f0(snd):
    """F0 mean/std/min/max/range, voicing_ratio, voice_breaks — 7 features."""
    try:
        pitch   = snd.to_pitch(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
        f0_arr  = pitch.selected_array["frequency"]
        voiced  = f0_arr[f0_arr > 0]
        n_total = len(f0_arr)

        is_voiced    = (f0_arr > 0).astype(int)
        voice_breaks = int(np.sum(np.diff(is_voiced) == -1))

        if len(voiced) == 0:
            return {k: np.nan for k in
                    ["f0_mean","f0_std","f0_min","f0_max","f0_range",
                     "voicing_ratio","voice_breaks"]}
        return dict(
            f0_mean       = float(np.mean(voiced)),
            f0_std        = float(np.std(voiced)),
            f0_min        = float(np.min(voiced)),
            f0_max        = float(np.max(voiced)),
            f0_range      = float(np.max(voiced) - np.min(voiced)),
            voicing_ratio = float(len(voiced) / max(n_total, 1)),
            voice_breaks  = float(voice_breaks),
        )
    except Exception:
        return {k: np.nan for k in
                ["f0_mean","f0_std","f0_min","f0_max","f0_range",
                 "voicing_ratio","voice_breaks"]}


def _formant_bandwidths(snd):
    """Formant bandwidths B1/B2/B3 (mean, std) — 6 features."""
    try:
        f  = snd.to_formant_burg(max_number_of_formants=MAX_FORMANTS,
                                  maximum_formant=MAX_FREQ)
        bv = {n: [] for n in range(1, 4)}
        for t in f.xs():
            for n in range(1, 4):
                b = call(f, "Get bandwidth at time", n, t, "Hertz", "Linear")
                if np.isfinite(b) and b > 0:
                    bv[n].append(b)
        out = {}
        for n in range(1, 4):
            a = np.array(bv[n])
            out[f"B{n}_mean"] = float(np.mean(a)) if len(a) else np.nan
            out[f"B{n}_std"]  = float(np.std(a))  if len(a) else np.nan
        return out
    except Exception:
        return {f"B{n}_{s}": np.nan for n in range(1, 4) for s in ("mean", "std")}


def _cpp(y):
    """Cepstral Peak Prominence (mean, std over frames) — 2 features."""
    try:
        frame_len = int(0.025 * SR)
        hop_len   = int(0.010 * SR)
        q_min     = max(1, int(SR / F0_MAX))
        q_max     = min(frame_len // 2 - 1, int(SR / F0_MIN))

        frames    = librosa.util.frame(y, frame_length=frame_len,
                                       hop_length=hop_len, axis=0)
        win       = np.hanning(frame_len)
        cpp_vals  = []
        for frame in frames:
            S_log    = np.log(np.abs(np.fft.rfft(frame * win)) ** 2 + 1e-9)
            cep      = np.abs(np.fft.irfft(S_log))[:frame_len // 2]
            peak_i   = q_min + np.argmax(cep[q_min:q_max + 1])
            peak_v   = cep[peak_i]
            x        = np.arange(len(cep))
            slope, intercept = np.polyfit(x, cep, 1)
            cpp_vals.append(peak_v - (slope * peak_i + intercept))

        if cpp_vals:
            return dict(cpp_mean=float(np.mean(cpp_vals)),
                        cpp_std =float(np.std(cpp_vals)))
        return dict(cpp_mean=np.nan, cpp_std=np.nan)
    except Exception:
        return dict(cpp_mean=np.nan, cpp_std=np.nan)


def _spectral_tilt(S):
    """Spectral envelope slope and alpha ratio — 2 features."""
    freqs    = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
    mag_mean = np.mean(S, axis=1)

    mask     = freqs > 0
    slope, _ = np.polyfit(np.log(freqs[mask] + 1e-9),
                           np.log(mag_mean[mask] + 1e-9), 1)

    idx_1k   = int(np.searchsorted(freqs, 1000))
    alpha    = float(np.sum(mag_mean[:idx_1k]) /
                     (np.sum(mag_mean[idx_1k:]) + 1e-9))

    return dict(spectral_slope=float(slope), alpha_ratio=alpha)


# ═══════════════════════════════════════════════════════════════════════════════
# 4. FEATS_4 (8) — syllables / repetition / spec_entropy
# ═══════════════════════════════════════════════════════════════════════════════

def _syllables(y):
    env = np.abs(ssignal.hilbert(y))
    env_smooth = ssignal.medfilt(env, kernel_size=7)
    threshold  = np.mean(env_smooth) * 1.2
    active     = env_smooth > threshold

    segments, start = [], None
    for i, val in enumerate(active):
        if val and start is None:
            start = i
        elif not val and start is not None:
            segments.append((start, i))
            start = None
    if start is not None:
        segments.append((start, len(env_smooth)))

    if len(segments) < 2:
        return dict(syllable_count=len(segments),
                    syllable_duration_std=0.0,
                    syllable_interval_std=0.0,
                    syllable_rate=float(len(segments)))

    durations  = [(e - s) / SR for s, e in segments]
    intervals  = [(segments[i+1][0] - segments[i][1]) / SR
                  for i in range(len(segments) - 1)]
    return dict(
        syllable_count        = len(segments),
        syllable_duration_std = float(np.std(durations)),
        syllable_interval_std = float(np.std(intervals)),
        syllable_rate         = float(len(segments) / (segments[-1][1] / SR)),
    )


def _repetition(rms):
    acf  = np.correlate(rms, rms, mode='full')
    acf  = acf[len(acf) // 2:]
    peaks, _ = find_peaks(acf, height=np.max(acf) * 0.3)
    return dict(
        repetition_peak_count    = int(len(peaks)),
        repetition_strength_std  = float(np.std(acf[peaks])) if len(peaks) > 0 else 0.0,
    )


def _spec_entropy_variability(S):
    ps         = S / (np.sum(S, axis=0, keepdims=True) + 1e-9)
    spec_entr  = entropy(ps, axis=0)
    dx         = np.diff(spec_entr)
    return dict(
        spec_entropy_std       = float(np.std(spec_entr)),
        spec_entropy_delta_std = float(np.std(dx)),
    )


# ═══════════════════════════════════════════════════════════════════════════════
# 5. EXT TIER-1 (17) — BER / Hammarberg / loudness / H1-H2-A1-A3 / flatness
# ═══════════════════════════════════════════════════════════════════════════════

def _band_energy_ratios(S):
    """5 band energy ratios (eGeMAPS edges) of the time-averaged magnitude spectrum."""
    freqs    = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
    mag_mean = np.mean(S, axis=1)
    total    = float(np.sum(mag_mean) + 1e-9)
    out = {}
    for lo, hi in zip(BAND_EDGES[:-1], BAND_EDGES[1:]):
        idx = (freqs >= lo) & (freqs < hi)
        out[f"ber_{lo}_{hi}"] = float(np.sum(mag_mean[idx]) / total)
    return out


def _hammarberg(S):
    """Hammarberg index (dB): max{0–2 kHz} / max{2–5 kHz} — hoarseness marker."""
    freqs    = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
    mag_mean = np.mean(S, axis=1)
    idx_2k   = int(np.searchsorted(freqs, 2000))
    idx_5k   = int(np.searchsorted(freqs, 5000))
    low_max  = float(np.max(mag_mean[:idx_2k]))      if idx_2k > 0       else 0.0
    high_max = float(np.max(mag_mean[idx_2k:idx_5k])) if idx_5k > idx_2k else 1e-9
    return dict(hammarberg_db=float(20.0 * np.log10((low_max + 1e-9) / (high_max + 1e-9))))


def _loudness_db(y):
    """RMS loudness in dB: mean / std / range (P95–P5) — 3 features."""
    rms_frames = librosa.feature.rms(y=y, hop_length=HOP)[0]
    rms_db = 20.0 * np.log10(rms_frames + 1e-9)
    return dict(
        loudness_db_mean  = float(np.mean(rms_db)),
        loudness_db_std   = float(np.std(rms_db)),
        loudness_db_range = float(np.percentile(rms_db, 95) - np.percentile(rms_db, 5)),
    )


def _glottal_source(y, snd):
    """H1-H2 / H1-A1 / H1-A3 (mean, std over voiced frames) — 6 features.
    H1 = amplitude at F0, H2 = at 2·F0, A1/A3 = at the bin nearest F1/F3 (dB)."""
    keys_nan = {f"{k}_{s}": np.nan
                for k in ("h1h2","h1a1","h1a3") for s in ("mean","std")}
    try:
        pitch   = snd.to_pitch(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
        f0_arr  = pitch.selected_array["frequency"]
        p_times = pitch.xs()

        formant = snd.to_formant_burg(max_number_of_formants=MAX_FORMANTS,
                                       maximum_formant=MAX_FREQ)

        spec_fine = np.abs(librosa.stft(y.astype(np.float32),
                                  n_fft=N_FFT_FINE, hop_length=HOP))
        spec_fine_db   = 20.0 * np.log10(spec_fine + 1e-9)
        fine_freqs = librosa.fft_frequencies(sr=SR, n_fft=N_FFT_FINE)
        frame_times = librosa.frames_to_time(np.arange(spec_fine_db.shape[1]),
                                          sr=SR, hop_length=HOP)

        def _db_at(spec, f):
            if f <= 0 or f >= SR / 2:
                return None
            return float(spec[int(np.argmin(np.abs(fine_freqs - f)))])

        h1h2, h1a1, h1a3 = [], [], []
        for t, f0 in zip(p_times, f0_arr):
            if f0 <= 0:
                continue
            frame_idx = int(np.argmin(np.abs(frame_times - t)))
            spec = spec_fine_db[:, frame_idx]
            f1 = formant.get_value_at_time(1, t)
            f3 = formant.get_value_at_time(3, t)
            h1 = _db_at(spec, f0)
            h2 = _db_at(spec, 2 * f0)
            a1 = _db_at(spec, f1) if np.isfinite(f1) else None
            a3 = _db_at(spec, f3) if np.isfinite(f3) else None
            if h1 is not None and h2 is not None: h1h2.append(h1 - h2)
            if h1 is not None and a1 is not None: h1a1.append(h1 - a1)
            if h1 is not None and a3 is not None: h1a3.append(h1 - a3)

        def _stats(vals):
            if not vals:
                return np.nan, np.nan
            return float(np.mean(vals)), float(np.std(vals))

        h1h2_m, h1h2_s = _stats(h1h2)
        h1a1_m, h1a1_s = _stats(h1a1)
        h1a3_m, h1a3_s = _stats(h1a3)
        return dict(h1h2_mean=h1h2_m, h1h2_std=h1h2_s,
                    h1a1_mean=h1a1_m, h1a1_std=h1a1_s,
                    h1a3_mean=h1a3_m, h1a3_std=h1a3_s)
    except Exception:
        return keys_nan


def _spectral_flatness(y):
    """Wiener entropy (spectral flatness) — 2 features (mean, std)."""
    try:
        flat = librosa.feature.spectral_flatness(y=y.astype(np.float32),
                                                  n_fft=N_FFT, hop_length=HOP)[0]
        return dict(flatness_mean=float(np.mean(flat)),
                    flatness_std =float(np.std(flat)))
    except Exception:
        return dict(flatness_mean=np.nan, flatness_std=np.nan)


# ═══════════════════════════════════════════════════════════════════════════════
# 6. EXT TIER-2 (11) — CPPS / GNE / F0 contour / pause-rate
# ═══════════════════════════════════════════════════════════════════════════════

def _cpps(y):
    """CPP Smoothed (Hillenbrand): 2D moving average over the cepstrogram — 2 features."""
    try:
        frame_len = int(0.025 * SR)
        hop_len   = int(0.010 * SR)
        q_min = max(1, int(SR / F0_MAX))
        q_max = min(frame_len // 2 - 1, int(SR / F0_MIN))

        frames = librosa.util.frame(y, frame_length=frame_len,
                                     hop_length=hop_len, axis=0)
        win    = np.hanning(frame_len)

        ceps = []
        for fr in frames:
            S_log = np.log(np.abs(np.fft.rfft(fr * win)) ** 2 + 1e-9)
            cep   = np.abs(np.fft.irfft(S_log))[:frame_len // 2]
            ceps.append(cep)
        if not ceps:
            return dict(cpps_mean=np.nan, cpps_std=np.nan)
        ceps = np.asarray(ceps)

        # Hillenbrand smoothing: ~10 ms in time × ~1 ms in quefrency
        ceps_s = uniform_filter(ceps, size=(3, 11), mode="nearest")

        cpps_vals = []
        for cep in ceps_s:
            peak_i = q_min + int(np.argmax(cep[q_min:q_max + 1]))
            peak_v = float(cep[peak_i])
            x = np.arange(len(cep))
            slope, intercept = np.polyfit(x, cep, 1)
            cpps_vals.append(peak_v - (slope * peak_i + intercept))

        return dict(cpps_mean=float(np.mean(cpps_vals)),
                    cpps_std =float(np.std(cpps_vals)))
    except Exception:
        return dict(cpps_mean=np.nan, cpps_std=np.nan)


def _gne(y):
    """Glottal-to-Noise Excitation (Michaelis 1997) — 2 features.
    Bandpass → Hilbert envelope → max cross-correlation across band pairs."""
    try:
        bands = [(500, 1500), (1000, 2000), (1500, 2500), (2000, 3000)]
        seg_len = SR  # 1 s segments
        n_segs  = max(1, len(y) // seg_len)
        gne_vals = []
        for s in range(n_segs):
            y_seg = y[s * seg_len:(s + 1) * seg_len]
            if len(y_seg) < SR // 2:
                continue
            envs = []
            for lo, hi in bands:
                b, a = butter(4, [lo / (SR / 2), hi / (SR / 2)], btype="band")
                yf = filtfilt(b, a, y_seg)
                envs.append(np.abs(hilbert(yf)))
            cors = []
            for i in range(len(envs)):
                for j in range(i + 1, len(envs)):
                    e1 = envs[i] - np.mean(envs[i])
                    e2 = envs[j] - np.mean(envs[j])
                    n  = np.sqrt(np.sum(e1 ** 2) * np.sum(e2 ** 2))
                    if n > 0:
                        c = float(np.max(np.correlate(e1, e2, mode="full")) / n)
                        cors.append(c)
            if cors:
                gne_vals.append(np.max(cors))
        if not gne_vals:
            return dict(gne_mean=np.nan, gne_std=np.nan)
        return dict(gne_mean=float(np.mean(gne_vals)),
                    gne_std =float(np.std(gne_vals)))
    except Exception:
        return dict(gne_mean=np.nan, gne_std=np.nan)


def _f0_contour(snd):
    """F0 slope per voiced segment + trajectory entropy — 3 features."""
    keys_nan = dict(f0_slope_mean=np.nan, f0_slope_std=np.nan,
                    f0_traj_entropy=np.nan)
    try:
        pitch  = snd.to_pitch(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
        f0_arr = pitch.selected_array["frequency"]
        times  = pitch.xs()
        if len(times) < 2:
            return keys_nan
        dt = float(times[1] - times[0])

        is_v = f0_arr > 0
        slopes = []
        i = 0
        while i < len(is_v):
            if is_v[i]:
                j = i
                while j < len(is_v) and is_v[j]:
                    j += 1
                seg = f0_arr[i:j]
                if len(seg) >= 5:
                    x = np.arange(len(seg)) * dt
                    m, _ = np.polyfit(x, seg, 1)
                    slopes.append(float(m))
                i = j
            else:
                i += 1

        voiced = f0_arr[is_v]
        if len(voiced) >= 10:
            df0 = np.diff(voiced)
            hist, _ = np.histogram(df0, bins=20, density=True)
            traj_ent = float(entropy(hist + 1e-9))
        else:
            traj_ent = np.nan

        return dict(
            f0_slope_mean   = float(np.mean(slopes)) if slopes else np.nan,
            f0_slope_std    = float(np.std(slopes))  if slopes else np.nan,
            f0_traj_entropy = traj_ent,
        )
    except Exception:
        return keys_nan


def _pause_rate(y, snd):
    """Pause count, mean pause dur, articulation rate, voiced ratio — 4 features."""
    try:
        pitch  = snd.to_pitch(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
        f0_arr = pitch.selected_array["frequency"]
        times  = pitch.xs()
        if len(times) < 2:
            return dict(pause_count=0, pause_mean_dur=0.0,
                        articulation_rate=np.nan, voiced_ratio=np.nan)
        dt = float(times[1] - times[0])
        is_v = f0_arr > 0

        min_pause_frames = max(1, int(0.2 / dt))   # 200 ms threshold
        pauses = []
        i = 0
        while i < len(is_v):
            if not is_v[i]:
                j = i
                while j < len(is_v) and not is_v[j]:
                    j += 1
                if (j - i) >= min_pause_frames:
                    pauses.append((j - i) * dt)
                i = j
            else:
                i += 1

        total_dur     = len(y) / SR
        voiced_ratio  = float(np.sum(is_v)) / max(1, len(is_v))
        # articulation_rate: voiced frames / total non-pause time
        non_pause_dur = max(1e-9, total_dur - sum(pauses))
        artic_rate    = float((np.sum(is_v) * dt) / non_pause_dur)

        return dict(
            pause_count       = int(len(pauses)),
            pause_mean_dur    = float(np.mean(pauses)) if pauses else 0.0,
            articulation_rate = artic_rate,
            voiced_ratio      = float(voiced_ratio),
        )
    except Exception:
        return dict(pause_count=0, pause_mean_dur=0.0,
                    articulation_rate=np.nan, voiced_ratio=np.nan)


# ═══════════════════════════════════════════════════════════════════════════════
# 7. EXT TIER-3 (9) — DFA / PPE / SHR / LPC residual / SPI / VTI
# ═══════════════════════════════════════════════════════════════════════════════

def _dfa(snd):
    """Detrended Fluctuation Analysis of the F0 contour — 1 feature (α)."""
    try:
        pitch  = snd.to_pitch(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
        f0_arr = pitch.selected_array["frequency"]
        voiced = f0_arr[f0_arr > 0]
        if len(voiced) < 50:
            return dict(dfa_alpha=np.nan)

        x = np.cumsum(voiced - np.mean(voiced))
        scales = np.unique(np.logspace(1, np.log10(max(10, len(x) // 4)), 15).astype(int))
        F = []
        used = []
        for s in scales:
            n_segs = len(x) // s
            if n_segs < 2:
                continue
            rms = []
            for i in range(n_segs):
                seg = x[i * s:(i + 1) * s]
                t   = np.arange(s)
                p   = np.polyfit(t, seg, 1)
                detr = seg - (p[0] * t + p[1])
                rms.append(np.sqrt(np.mean(detr ** 2)))
            F.append(np.mean(rms))
            used.append(s)
        if len(F) < 3:
            return dict(dfa_alpha=np.nan)
        log_s = np.log(np.asarray(used))
        log_F = np.log(np.asarray(F) + 1e-9)
        alpha, _ = np.polyfit(log_s, log_F, 1)
        return dict(dfa_alpha=float(alpha))
    except Exception:
        return dict(dfa_alpha=np.nan)


def _ppe(snd):
    """Pitch Period Entropy — 1 feature (normalised entropy of the log-F0 residual)."""
    try:
        pitch  = snd.to_pitch(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
        f0_arr = pitch.selected_array["frequency"]
        voiced = f0_arr[f0_arr > 0]
        if len(voiced) < 30:
            return dict(ppe=np.nan)
        log_f0 = np.log(voiced)
        # whitening: AR(1) prediction residual
        x  = log_f0[1:]
        x0 = log_f0[:-1]
        rho = float(np.corrcoef(x, x0)[0, 1]) if len(x) > 1 else 0.0
        res = x - rho * x0
        n_bins = 30
        hist, _ = np.histogram(res, bins=n_bins, density=True)
        hist = hist + 1e-9
        h = float(entropy(hist) / np.log(n_bins))
        return dict(ppe=h)
    except Exception:
        return dict(ppe=np.nan)


def _shr(y, snd):
    """Subharmonic-to-Harmonic Ratio — 2 features (mean, std over voiced frames)."""
    try:
        pitch  = snd.to_pitch(pitch_floor=F0_MIN, pitch_ceiling=F0_MAX)
        f0_arr = pitch.selected_array["frequency"]
        p_times = pitch.xs()
        if len(p_times) < 2:
            return dict(shr_mean=np.nan, shr_std=np.nan)

        spec_fine = np.abs(librosa.stft(y.astype(np.float32),
                                  n_fft=N_FFT_FINE, hop_length=HOP))
        fine_freqs = librosa.fft_frequencies(sr=SR, n_fft=N_FFT_FINE)
        frame_times = librosa.frames_to_time(np.arange(spec_fine.shape[1]),
                                          sr=SR, hop_length=HOP)

        shrs = []
        for t, f0 in zip(p_times, f0_arr):
            if f0 <= 0 or 2 * f0 >= SR / 2:
                continue
            frame_idx = int(np.argmin(np.abs(frame_times - t)))
            spec = spec_fine[:, frame_idx]
            even_amp = 0.0
            odd_amp  = 0.0
            for k in range(1, 6):
                fh = k * f0
                if fh >= SR / 2:
                    break
                even_amp += float(spec[int(np.argmin(np.abs(fine_freqs - fh)))])
                fsh = (k - 0.5) * f0
                if 0 < fsh < SR / 2:
                    odd_amp += float(spec[int(np.argmin(np.abs(fine_freqs - fsh)))])
            if even_amp > 0:
                shrs.append(odd_amp / (even_amp + 1e-9))
        if not shrs:
            return dict(shr_mean=np.nan, shr_std=np.nan)
        return dict(shr_mean=float(np.mean(shrs)),
                    shr_std =float(np.std(shrs)))
    except Exception:
        return dict(shr_mean=np.nan, shr_std=np.nan)


def _lpc_residual(y):
    """LPC(16) residual energy + residual periodicity (HNR-like) — 3 features."""
    try:
        order     = 16
        frame_len = int(0.025 * SR)
        hop_len   = int(0.010 * SR)
        frames = librosa.util.frame(y, frame_length=frame_len,
                                     hop_length=hop_len, axis=0)
        win    = np.hanning(frame_len)

        rms_vals = []
        hnr_vals = []
        min_lag  = int(SR / F0_MAX)
        max_lag  = int(SR / F0_MIN)

        for fr in frames:
            fr_w = fr * win
            try:
                a = librosa.lpc(fr_w, order=order)
            except Exception:
                continue
            res = ssignal.lfilter(a, [1.0], fr_w)
            r0  = float(np.mean(res ** 2))
            if r0 <= 0:
                continue
            rms_vals.append(np.sqrt(r0))
            acf = np.correlate(res, res, mode="full")
            acf = acf[len(acf) // 2:]
            if max_lag > min_lag and max_lag < len(acf) and acf[0] > 0:
                peak = float(np.max(acf[min_lag:max_lag]))
                hnr_vals.append(peak / acf[0])

        return dict(
            lpc_res_rms_mean = float(np.mean(rms_vals)) if rms_vals else np.nan,
            lpc_res_rms_std  = float(np.std(rms_vals))  if rms_vals else np.nan,
            lpc_res_hnr_mean = float(np.mean(hnr_vals)) if hnr_vals else np.nan,
        )
    except Exception:
        return dict(lpc_res_rms_mean=np.nan, lpc_res_rms_std=np.nan,
                    lpc_res_hnr_mean=np.nan)


def _spi(S):
    """Soft Phonation Index: energy 70–1600 Hz / energy 1600–4500 Hz — 1 feature."""
    try:
        freqs    = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
        mag_mean = np.mean(S, axis=1)
        low      = (freqs >= 70)   & (freqs < 1600)
        high     = (freqs >= 1600) & (freqs < 4500)
        low_e    = float(np.sum(mag_mean[low]))
        high_e   = float(np.sum(mag_mean[high])) + 1e-9
        return dict(spi=float(low_e / high_e))
    except Exception:
        return dict(spi=np.nan)


def _vti(S):
    """Voice Turbulence Index: noise 2800–5800 Hz / signal 70–4500 Hz — 1 feature."""
    try:
        freqs    = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
        mag_mean = np.mean(S, axis=1)
        sig      = (freqs >= 70)   & (freqs < 4500)
        noise    = (freqs >= 2800) & (freqs < 5800)
        sig_e    = float(np.sum(mag_mean[sig])) + 1e-9
        noise_e  = float(np.sum(mag_mean[noise]))
        return dict(vti=float(noise_e / sig_e))
    except Exception:
        return dict(vti=np.nan)


N_MFCC_EXT       = 20
N_CONTRAST_BANDS = 7   # → 8 rows in the spectral_contrast output


def _mean_std_var(vec, name):
    """mean/std/var of a vector, paper-style names (`{stat}_{name}`)."""
    arr = np.asarray(vec, dtype=float)
    if arr.size == 0 or not np.any(np.isfinite(arr)):
        return {f"mean_{name}": np.nan,
                f"std_{name}":  np.nan,
                f"var_{name}":  np.nan}
    return {f"mean_{name}": float(np.mean(arr)),
            f"std_{name}":  float(np.std(arr)),
            f"var_{name}":  float(np.var(arr))}


def _mean_std_var_nan(name):
    return {f"mean_{name}": np.nan,
            f"std_{name}":  np.nan,
            f"var_{name}":  np.nan}


def _aux_features(y, S):
    """277 paper-style features: mean/std/var of every parameter."""
    out = {}
    y_f32 = y.astype(np.float32)

    # ── MFCC, MFCC∆, MFCC∆∆  (20 × 3 stats × 3 series = 180 features)
    try:
        mfcc20 = librosa.feature.mfcc(y=y_f32, sr=SR, n_mfcc=N_MFCC_EXT)
        d1     = librosa.feature.delta(mfcc20, order=1)
        d2     = librosa.feature.delta(mfcc20, order=2)
        for n in range(1, N_MFCC_EXT + 1):
            out.update(_mean_std_var(mfcc20[n - 1], f"mfcc{n}"))
            out.update(_mean_std_var(d1[n - 1],     f"mfcc_delta_{n}"))
            out.update(_mean_std_var(d2[n - 1],     f"mfcc_delta2_{n}"))
    except Exception:
        for n in range(1, N_MFCC_EXT + 1):
            out.update(_mean_std_var_nan(f"mfcc{n}"))
            out.update(_mean_std_var_nan(f"mfcc_delta_{n}"))
            out.update(_mean_std_var_nan(f"mfcc_delta2_{n}"))

    # ── Chroma CENS (12 bins × 3 = 36 features)
    try:
        cens = librosa.feature.chroma_cens(y=y_f32, sr=SR, hop_length=HOP)
        for n in range(1, 13):
            out.update(_mean_std_var(cens[n - 1], f"chroma_cens_{n}"))
    except Exception:
        for n in range(1, 13):
            out.update(_mean_std_var_nan(f"chroma_cens_{n}"))

    # ── Tonnetz (6 dims × 3 = 18 features)
    try:
        y_harm = librosa.effects.harmonic(y_f32)
        tonn   = librosa.feature.tonnetz(y=y_harm, sr=SR)
        n_dims = tonn.shape[0]
        for n in range(1, n_dims + 1):
            out.update(_mean_std_var(tonn[n - 1], f"tonnetz_{n}"))
        for n in range(n_dims + 1, 7):  # pad to 6 if fewer
            out.update(_mean_std_var_nan(f"tonnetz_{n}"))
    except Exception:
        for n in range(1, 7):
            out.update(_mean_std_var_nan(f"tonnetz_{n}"))

    # ── Spectral contrast (8 bands × 3 = 24 features)
    try:
        sc = librosa.feature.spectral_contrast(S=S, sr=SR,
                                                n_bands=N_CONTRAST_BANDS)
        n_bands = sc.shape[0]  # = N_CONTRAST_BANDS + 1 = 8
        for n in range(1, n_bands + 1):
            out.update(_mean_std_var(sc[n - 1], f"spectral_contrast_{n}"))
        for n in range(n_bands + 1, N_CONTRAST_BANDS + 2):
            out.update(_mean_std_var_nan(f"spectral_contrast_{n}"))
    except Exception:
        for n in range(1, N_CONTRAST_BANDS + 2):
            out.update(_mean_std_var_nan(f"spectral_contrast_{n}"))

    # ── Scalars (6 parameters × 3 = 18 features)
    try:
        out.update(_mean_std_var(librosa.feature.rms(y=y, hop_length=HOP)[0], "rms"))
    except Exception:
        out.update(_mean_std_var_nan("rms"))
    try:
        out.update(_mean_std_var(librosa.feature.spectral_centroid(S=S, sr=SR)[0],
                        "spectral_centroid"))
    except Exception:
        out.update(_mean_std_var_nan("spectral_centroid"))
    try:
        out.update(_mean_std_var(librosa.feature.spectral_bandwidth(S=S, sr=SR)[0],
                        "spectral_bandwidth"))
    except Exception:
        out.update(_mean_std_var_nan("spectral_bandwidth"))
    try:
        out.update(_mean_std_var(librosa.feature.spectral_rolloff(S=S, sr=SR)[0],
                        "spectral_rolloff"))
    except Exception:
        out.update(_mean_std_var_nan("spectral_rolloff"))
    try:
        out.update(_mean_std_var(librosa.feature.spectral_flatness(y=y_f32,
                                                          n_fft=N_FFT,
                                                          hop_length=HOP)[0],
                        "spectral_flatness"))
    except Exception:
        out.update(_mean_std_var_nan("spectral_flatness"))
    try:
        out.update(_mean_std_var(librosa.feature.zero_crossing_rate(y, hop_length=HOP)[0],
                        "zcr"))
    except Exception:
        out.update(_mean_std_var_nan("zcr"))

    # ── Tempo (1 scalar — no mean/std/var)
    try:
        tempo_val, _ = librosa.beat.beat_track(y=y_f32, sr=SR)
        out["tempo"] = float(np.atleast_1d(tempo_val).flatten()[0])
    except Exception:
        out["tempo"] = np.nan

    return out

def _extract_from_signal(y_win: np.ndarray) -> dict:
    """All 464 features of an already loaded signal."""
    snd = parselmouth.Sound(y_win.astype(np.float64), sampling_frequency=SR)
    S   = np.abs(librosa.stft(y_win, n_fft=N_FFT, hop_length=HOP))

    out = {}
    out.update(_jitter(snd))
    out.update(_shimmer(snd))
    out.update(_hnr(snd))
    out.update(_formants(snd))
    out.update(_mfcc(y_win))
    direct, rms = _direct(y_win, S)
    out.update(direct)
    out.update(_syllables(y_win))
    out.update(_repetition(rms))
    out.update(_spec_entropy_variability(S))
    out.update(_f0(snd))
    out.update(_formant_bandwidths(snd))
    out.update(_cpp(y_win))
    out.update(_spectral_tilt(S))

    out.update(_band_energy_ratios(S))
    out.update(_hammarberg(S))
    out.update(_loudness_db(y_win))
    out.update(_glottal_source(y_win, snd))
    out.update(_spectral_flatness(y_win))

    out.update(_cpps(y_win))
    out.update(_gne(y_win))
    out.update(_f0_contour(snd))
    out.update(_pause_rate(y_win, snd))

    out.update(_dfa(snd))
    out.update(_ppe(snd))
    out.update(_shr(y_win, snd))
    out.update(_lpc_residual(y_win))
    out.update(_spi(S))
    out.update(_vti(S))

    out.update(_aux_features(y_win, S))
    return out


def extract_features(path: str, sec: float = 1.0) -> dict:
    """464 features of the first `sec` seconds of an audio file.

    Only the prefix is decoded, so long recordings cost the same as short ones.
    Returns a flat {feature_name: float} dict. Short prefixes (< 2 s) leave many
    Tier-3 features (DFA, SHR, tempo) as NaN — they need a longer signal.
    """
    y, _ = librosa.load(path, sr=SR, mono=True, duration=sec)
    if len(y) == 0:
        raise ValueError(f"Loaded 0 samples from '{path}' (sec={sec}).")
    y = y / (np.max(np.abs(y)) + 1e-9)
    return _extract_from_signal(y)

## build features table


In [ ]:
SEC = 5.0
N_PROC = 16
PROGRESS_EVERY = 200
CHECKPOINT_EVERY = 1000
CHECKPOINT_PATH = FEATURES_PARQUET.with_suffix(".ckpt.pkl")
FAILURE_LOG = FEATURES_PARQUET.with_suffix(".failures.csv")
META_COLS = ["VoxCeleb_ID", "video_id", "gender", "speaker_age", "wav_path"]


def extract_one(task):
    row_idx, wav_path = task
    try:
        with threadpool_limits(limits=1):
            features = extract_features(wav_path, SEC)
        return row_idx, features, None
    except Exception as e:
        return row_idx, None, f"{type(e).__name__}: {e}"


def build_feature_table():
    recordings = pd.read_csv(RECORDINGS_CSV)
    recordings["wav_path"] = recordings["dest"].str.replace(r"\.m4a$", ".wav", regex=True)
    n_total = len(recordings)
    print(f"[load] {RECORDINGS_CSV}  rows: {n_total}  (SEC={SEC})", flush=True)

    rows, done = [], set()
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "rb") as f:
            rows = pickle.load(f)
        done = {r["__row_idx"] for r in rows}
        print(f"[resume] {len(done)} rows from checkpoint", flush=True)

    todo = [(i, p) for i, p in enumerate(recordings["wav_path"]) if i not in done]
    print(f"[todo] {len(todo)} / {n_total} files", flush=True)

    t_start = time.time()
    n_done, n_failed = len(done), 0
    with open(FAILURE_LOG, "a", newline="") as fail_file:
        fail_log = csv.writer(fail_file)
        if fail_file.tell() == 0:
            fail_log.writerow(["row_idx", "wav_path", "error"])
        if todo:
            with mp.Pool(N_PROC) as pool:
                for row_idx, features, error in pool.imap_unordered(extract_one, todo, chunksize=4):
                    n_done += 1
                    if error:
                        n_failed += 1
                        fail_log.writerow([row_idx, recordings["wav_path"].iloc[row_idx], error])
                        fail_file.flush()
                    else:
                        features["__row_idx"] = row_idx
                        rows.append(features)
                    if n_done % PROGRESS_EVERY == 0 or n_done == n_total:
                        elapsed = time.time() - t_start
                        print(f"[sec={SEC:g} {n_done:6d}/{n_total}] fail={n_failed} "
                              f"elapsed={elapsed/60:5.1f}min", flush=True)
                    if n_done % CHECKPOINT_EVERY == 0:
                        with open(CHECKPOINT_PATH, "wb") as f:
                            pickle.dump(rows, f)

    features = pd.DataFrame(rows).set_index("__row_idx").sort_index()
    table = recordings[META_COLS].join(features, how="inner")
    table.to_parquet(FEATURES_PARQUET, compression="zstd")
    print(f"[save] {FEATURES_PARQUET}  shape={table.shape}  failed={n_failed}", flush=True)
    if CHECKPOINT_PATH.exists():
        os.remove(CHECKPOINT_PATH)


if FEATURES_PARQUET.exists():
    print(f"exists, not recomputed: {FEATURES_PARQUET}")
else:
    build_feature_table()

In [3]:
features_df = pd.read_parquet(FEATURES_PARQUET)
print("full population:", features_df.shape)

features_df = features_df[features_df["gender"].isin(["male", "female"])].reset_index(drop=True)
print("after keeping male/female only:", features_df.shape)
print(features_df["gender"].value_counts())

full population: (47764, 469)
after keeping male/female only: (47763, 469)
gender
male      30174
female    17589
Name: count, dtype: int64


after keeping male/female only: (47763, 469)
gender
male      30174
female    17589
Name: count, dtype: int64


## Feature selection

### Drop 100%-NaN and zero-variance columns 

In [4]:
feature_cols = [c for c in features_df.columns if c not in META_COLS]

constant_cols = [c for c in feature_cols
                 if features_df[c].notna().sum() == 0 or features_df[c].std(skipna=True) == 0]
feature_cols = [c for c in feature_cols if c not in constant_cols]
print(f"dropped {len(constant_cols)} dead columns -> {len(feature_cols)} features")

dropped 24 dead columns -> 440 features


## Drop near-duplicate columns

In [5]:
REDUNDANCY_THRESHOLD = 0.99

abs_corr = features_df[feature_cols].corr(method="spearman").abs()
upper_triangle = abs_corr.where(np.triu(np.ones(abs_corr.shape), k=1).astype(bool))
redundant_cols = [c for c in upper_triangle.columns if (upper_triangle[c] > REDUNDANCY_THRESHOLD).any()]
feature_cols = [c for c in feature_cols if c not in redundant_cols]
print(f"dropped {len(redundant_cols)} near-duplicate columns (Spearman > {REDUNDANCY_THRESHOLD}) "
      f"-> {len(feature_cols)} features")

dropped 179 near-duplicate columns (Spearman > 0.99) -> 261 features
